In [1]:
# -*- coding: utf-8 -*-
"""
LFSR 100-step simulator with Tkinter buttons and final statistics.

実行方法:
    python lfsr_100_tkinter_complete.py

主な機能:
- LFSRの状態を100クロック分シミュレーション
- レジスタ状態をアニメーション表示
- 再生、一時停止、前後移動
- 再生速度の変更
- 任意のクロック位置への移動
- 出力系列と統計情報の表示
- 新しい生成多項式と初期値の生成

注意:
- 自動再生はしません。
- PlayまたはRun 100を押してください。
"""

import random
import tkinter as tk
from tkinter import ttk

import matplotlib

# Tkinterウィンドウ内にMatplotlibを表示する
matplotlib.use("TkAgg")

import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.patches import Rectangle


# =========================================================
# 1. 基本設定
# =========================================================

STEPS = 100
DEFAULT_INTERVAL_MS = 500
MAX_SHOW_BITS = 64


# 原始多項式の候補
#
# 例:
# [4, 1, 0] は
# x^4 + x + 1
# を表す
PRIMITIVE_POLYNOMIALS = {
    4: [
        [4, 1, 0],
        [4, 3, 0],
    ],
    5: [
        [5, 2, 0],
        [5, 4, 2, 1, 0],
    ],
    6: [
        [6, 1, 0],
        [6, 5, 0],
    ],
    7: [
        [7, 1, 0],
        [7, 3, 0],
    ],
    8: [
        [8, 4, 3, 2, 0],
        [8, 6, 5, 4, 0],
    ],
    10: [
        [10, 3, 0],
        [10, 7, 0],
    ],
    12: [
        [12, 6, 4, 1, 0],
    ],
    16: [
        [16, 12, 3, 1, 0],
        [16, 5, 3, 2, 0],
    ],
    24: [
        [24, 4, 3, 1, 0],
    ],
    32: [
        [32, 22, 2, 1, 0],
    ],
    48: [
        [48, 28, 27, 1, 0],
    ],
    64: [
        [64, 4, 3, 1, 0],
    ],
}


# =========================================================
# 2. 補助関数
# =========================================================

def bits_to_str(bits):
    """ビット列を文字列に変換する。"""
    return "".join(str(bit) for bit in bits)


def polynomial_to_string(exponents):
    """
    多項式の指数リストを文字列に変換する。

    例:
    [4, 1, 0]
    ->
    x^4 + x + 1
    """
    terms = []

    for exponent in sorted(exponents, reverse=True):
        if exponent == 0:
            terms.append("1")
        elif exponent == 1:
            terms.append("x")
        else:
            terms.append(f"x^{exponent}")

    return " + ".join(terms)


def choose_random_primitive_polynomial():
    """原始多項式の候補から1つをランダムに選ぶ。"""
    degree = random.choice(list(PRIMITIVE_POLYNOMIALS.keys()))
    exponents = random.choice(PRIMITIVE_POLYNOMIALS[degree])

    return degree, exponents


def polynomial_to_taps(degree, exponents):
    """
    多項式の指数をレジスタ配列のタップ位置に変換する。

    レジスタは左から右へ
    R(n-1), R(n-2), ..., R1, R0
    と並んでいる。
    """
    taps = []

    for exponent in exponents:
        # 最高次数はレジスタの次数そのものなので除外
        if exponent == degree:
            continue

        if exponent == 0:
            taps.append(degree - 1)
        else:
            taps.append(degree - 1 - exponent)

    return sorted(set(taps))


def random_seed_bits(degree):
    """
    全ビット0ではないランダムな初期値を生成する。

    全ビット0は、LFSRが永久に0の状態になるため使用しない。
    """
    while True:
        seed = [
            random.randint(0, 1)
            for _ in range(degree)
        ]

        if any(seed):
            return seed


# =========================================================
# 3. LFSR本体
# =========================================================

class LFSR:
    """線形帰還シフトレジスタを表すクラス。"""

    def __init__(self, seed, taps):
        if not seed:
            raise ValueError("Seed must not be empty.")

        if all(bit == 0 for bit in seed):
            raise ValueError("Seed must not be all zero.")

        for bit in seed:
            if bit not in (0, 1):
                raise ValueError("Seed must contain only 0 and 1.")

        for tap in taps:
            if tap < 0 or tap >= len(seed):
                raise ValueError(f"Invalid tap index: {tap}")

        self.reg = seed[:]
        self.taps = taps[:]
        self.degree = len(seed)

    def step(self):
        """
        LFSRを1クロック進める。

        処理:
        1. タップ位置の値を取得
        2. XORでフィードバックを計算
        3. 右端ビットを出力
        4. フィードバックを左端に入力
        5. 全体を右へシフト
        """
        before = self.reg[:]

        tap_values = [
            before[tap]
            for tap in self.taps
        ]

        feedback = 0

        for value in tap_values:
            feedback ^= value

        output_bit = before[-1]

        # 左端にfeedbackを追加し、
        # 元の右端ビットを削除する
        self.reg = [feedback] + before[:-1]

        return {
            "before": before,
            "after": self.reg[:],
            "tap_values": tap_values,
            "feedback": feedback,
            "output_bit": output_bit,
        }


# =========================================================
# 4. 履歴の作成
# =========================================================

def make_history(seed, taps, steps):
    """
    指定されたクロック数だけLFSRを進め、
    全履歴を保存する。
    """
    lfsr = LFSR(seed, taps)

    # states[0]は初期状態
    states = [seed[:]]

    # クロック0ではfeedbackやoutputは存在しない
    feedbacks = [None]
    outputs_by_clock = [None]
    tap_values_by_clock = [None]

    output_sequence = []

    for _ in range(steps):
        info = lfsr.step()

        states.append(info["after"])
        feedbacks.append(info["feedback"])
        outputs_by_clock.append(info["output_bit"])
        tap_values_by_clock.append(info["tap_values"])
        output_sequence.append(info["output_bit"])

    return (
        states,
        feedbacks,
        outputs_by_clock,
        tap_values_by_clock,
        output_sequence,
    )


# =========================================================
# 5. ラン長の計算
# =========================================================

def run_lengths(bits):
    """
    連続する同じビットの長さを求める。

    例:
    001110011
    ->
    [(0, 2), (1, 3), (0, 2), (1, 2)]
    """
    if not bits:
        return []

    lengths = []

    current_bit = bits[0]
    count = 1

    for bit in bits[1:]:
        if bit == current_bit:
            count += 1
        else:
            lengths.append((current_bit, count))
            current_bit = bit
            count = 1

    lengths.append((current_bit, count))

    return lengths


# =========================================================
# 6. 統計計算
# =========================================================

def compute_stats(
    states,
    feedbacks,
    output_sequence,
    seed,
    theoretical_period,
):
    """LFSRの出力系列と状態履歴から統計量を計算する。"""

    valid_feedbacks = [
        value
        for value in feedbacks
        if value is not None
    ]

    output_bits = output_sequence[:]
    runs = run_lengths(output_bits)

    # 初期状態へ最初に戻ったクロックを探す
    first_return_clock = None

    for clock in range(1, len(states)):
        if states[clock] == seed:
            first_return_clock = clock
            break

    unique_states = len({
        tuple(state)
        for state in states
    })

    max_run = max(
        (length for _, length in runs),
        default=0,
    )

    if runs:
        avg_run = (
            sum(length for _, length in runs)
            / len(runs)
        )
    else:
        avg_run = 0.0

    zero_count = output_bits.count(0)
    one_count = output_bits.count(1)

    if output_bits:
        one_ratio = one_count / len(output_bits)
    else:
        one_ratio = 0.0

    return {
        "steps": len(output_bits),
        "output_0": zero_count,
        "output_1": one_count,
        "output_1_ratio": one_ratio,
        "feedback_0": valid_feedbacks.count(0),
        "feedback_1": valid_feedbacks.count(1),
        "unique_states": unique_states,
        "first_return_clock": first_return_clock,
        "theoretical_period": theoretical_period,
        "run_count": len(runs),
        "max_run": max_run,
        "avg_run": avg_run,
    }


# =========================================================
# 7. Tkinterアプリケーション
# =========================================================

class LFSRApp:

    def __init__(self, root):
        self.root = root

        self.root.title("LFSR 100-step Simulator")
        self.root.geometry("1150x760")
        self.root.minsize(900, 650)

        # ---------------------------------------------
        # LFSR問題を生成
        # ---------------------------------------------

        (
            self.degree,
            self.exponents,
        ) = choose_random_primitive_polynomial()

        self.taps = polynomial_to_taps(
            self.degree,
            self.exponents,
        )

        self.seed = random_seed_bits(self.degree)

        self.theoretical_period = (
            2 ** self.degree - 1
        )

        (
            self.states,
            self.feedbacks,
            self.outputs,
            self.tap_values,
            self.output_sequence,
        ) = make_history(
            self.seed,
            self.taps,
            STEPS,
        )

        # ---------------------------------------------
        # アニメーション状態
        # ---------------------------------------------

        self.index = 0
        self.playing = False
        self.after_id = None
        self.interval_ms = DEFAULT_INTERVAL_MS

        # スライダー操作中の再帰的な呼び出しを防ぐ
        self.updating_controls = False

        # ---------------------------------------------
        # GUI作成
        # ---------------------------------------------

        self._build_ui()

        self.draw()
        self.update_info()

    # =====================================================
    # GUIの作成
    # =====================================================

    def _build_ui(self):

        # ---------------------------------------------
        # 上部の情報表示
        # ---------------------------------------------

        top = ttk.Frame(
            self.root,
            padding=8,
        )
        top.pack(
            side=tk.TOP,
            fill=tk.X,
        )

        self.info_var = tk.StringVar()

        info_label = ttk.Label(
            top,
            textvariable=self.info_var,
            font=("Consolas", 10),
        )
        info_label.pack(
            side=tk.LEFT,
            fill=tk.X,
            expand=True,
        )

        # ---------------------------------------------
        # 操作ボタン
        # ---------------------------------------------

        controls = ttk.Frame(
            self.root,
            padding=8,
        )
        controls.pack(
            side=tk.TOP,
            fill=tk.X,
        )

        ttk.Button(
            controls,
            text="Play",
            command=self.play,
        ).pack(
            side=tk.LEFT,
            padx=4,
        )

        ttk.Button(
            controls,
            text="Pause",
            command=self.pause,
        ).pack(
            side=tk.LEFT,
            padx=4,
        )

        ttk.Button(
            controls,
            text="Prev",
            command=self.prev_step,
        ).pack(
            side=tk.LEFT,
            padx=4,
        )

        ttk.Button(
            controls,
            text="Next",
            command=self.next_step,
        ).pack(
            side=tk.LEFT,
            padx=4,
        )

        ttk.Button(
            controls,
            text="Reset",
            command=self.reset,
        ).pack(
            side=tk.LEFT,
            padx=4,
        )

        ttk.Button(
            controls,
            text="Run 100",
            command=self.run_to_end,
        ).pack(
            side=tk.LEFT,
            padx=4,
        )

        ttk.Button(
            controls,
            text="Stats",
            command=self.show_stats,
        ).pack(
            side=tk.LEFT,
            padx=4,
        )

        ttk.Button(
            controls,
            text="New",
            command=self.new_problem,
        ).pack(
            side=tk.LEFT,
            padx=4,
        )

        ttk.Button(
            controls,
            text="Quit",
            command=self.close,
        ).pack(
            side=tk.LEFT,
            padx=4,
        )

        # ---------------------------------------------
        # 速度スライダー
        # ---------------------------------------------

        ttk.Label(
            controls,
            text="  Speed(ms):",
        ).pack(
            side=tk.LEFT,
            padx=(20, 4),
        )

        # 重要:
        # Scale.set()でコールバックが呼ばれる可能性があるため、
        # speed_label_varを先に作成する
        self.speed_label_var = tk.StringVar(
            value=str(self.interval_ms)
        )

        self.speed_scale = ttk.Scale(
            controls,
            from_=100,
            to=2000,
            orient=tk.HORIZONTAL,
            command=self.on_speed_change,
            length=180,
        )

        self.speed_scale.pack(
            side=tk.LEFT,
            padx=4,
        )

        ttk.Label(
            controls,
            textvariable=self.speed_label_var,
            width=5,
        ).pack(
            side=tk.LEFT,
        )

        # 全GUI部品を作った後で値を設定
        self.speed_scale.set(self.interval_ms)

        # ---------------------------------------------
        # クロックスライダー
        # ---------------------------------------------

        ttk.Label(
            controls,
            text="  Clock:",
        ).pack(
            side=tk.LEFT,
            padx=(20, 4),
        )

        self.clock_label_var = tk.StringVar(
            value="0"
        )

        self.clock_scale = ttk.Scale(
            controls,
            from_=0,
            to=STEPS,
            orient=tk.HORIZONTAL,
            command=self.on_clock_change,
            length=260,
        )

        self.clock_scale.pack(
            side=tk.LEFT,
            padx=4,
        )

        ttk.Label(
            controls,
            textvariable=self.clock_label_var,
            width=4,
        ).pack(
            side=tk.LEFT,
        )

        self.clock_scale.set(0)

        # ---------------------------------------------
        # Matplotlib描画領域
        # ---------------------------------------------

        self.fig, self.ax = plt.subplots(
            figsize=(11.0, 4.7)
        )

        self.fig.subplots_adjust(
            left=0.04,
            right=0.98,
            top=0.86,
            bottom=0.18,
        )

        self.canvas = FigureCanvasTkAgg(
            self.fig,
            master=self.root,
        )

        self.canvas_widget = (
            self.canvas.get_tk_widget()
        )

        self.canvas_widget.pack(
            side=tk.TOP,
            fill=tk.BOTH,
            expand=True,
        )

        # ---------------------------------------------
        # 下部テキスト表示
        # ---------------------------------------------

        bottom = ttk.Frame(
            self.root,
            padding=8,
        )
        bottom.pack(
            side=tk.BOTTOM,
            fill=tk.X,
        )

        self.text = tk.Text(
            bottom,
            height=9,
            wrap=tk.WORD,
            font=("Consolas", 10),
        )

        self.text.pack(
            side=tk.LEFT,
            fill=tk.BOTH,
            expand=True,
        )

        scroll = ttk.Scrollbar(
            bottom,
            command=self.text.yview,
        )

        scroll.pack(
            side=tk.RIGHT,
            fill=tk.Y,
        )

        self.text.configure(
            yscrollcommand=scroll.set
        )

        # ---------------------------------------------
        # キーボード操作
        # ---------------------------------------------

        self.root.bind(
            "<space>",
            lambda event: self.toggle_play(),
        )

        self.root.bind(
            "<Left>",
            lambda event: self.prev_step(),
        )

        self.root.bind(
            "<Right>",
            lambda event: self.next_step(),
        )

        self.root.bind(
            "<Home>",
            lambda event: self.reset(),
        )

        self.root.bind(
            "<End>",
            lambda event: self.run_to_end(),
        )

        self.root.protocol(
            "WM_DELETE_WINDOW",
            self.close,
        )

    # =====================================================
    # スライダー処理
    # =====================================================

    def on_speed_change(self, value):
        """再生間隔を変更する。"""

        try:
            interval = int(float(value))
        except (TypeError, ValueError):
            return

        interval = max(100, min(interval, 2000))
        self.interval_ms = interval

        # 初期化途中でもエラーにならないように確認する
        if hasattr(self, "speed_label_var"):
            self.speed_label_var.set(
                str(self.interval_ms)
            )

        # 再生中に速度を変更した場合、
        # 次回更新を新しい速度で予約し直す
        if self.playing:
            self.schedule_next()

    def on_clock_change(self, value):
        """クロックスライダーの位置を変更する。"""

        if self.updating_controls:
            return

        # 再生中はクロックスライダー操作を無視する
        if self.playing:
            return

        try:
            index = int(round(float(value)))
        except (TypeError, ValueError):
            return

        if index != self.index:
            self.set_index(index)

    # =====================================================
    # 表示位置の変更
    # =====================================================

    def set_index(self, index):
        """表示するクロック位置を設定する。"""

        self.index = max(
            0,
            min(int(index), STEPS),
        )

        self.updating_controls = True

        try:
            self.clock_scale.set(self.index)
            self.clock_label_var.set(
                str(self.index)
            )
        finally:
            self.updating_controls = False

        self.draw()
        self.update_info()

    # =====================================================
    # LFSR図の描画
    # =====================================================

    def draw(self):

        self.ax.clear()

        register = self.states[self.index]

        # 最大64ビットまで表示
        shown_register = register[:MAX_SHOW_BITS]
        shown_count = len(shown_register)

        if len(register) > shown_count:
            note = (
                f"showing first {shown_count} "
                f"of {len(register)} bits"
            )
        else:
            note = f"{len(register)} bits"

        # ---------------------------------------------
        # 各レジスタを描画
        # ---------------------------------------------

        for index, bit in enumerate(shown_register):

            if bit == 1:
                face_color = "#BFBFBF"
            else:
                face_color = "white"

            rectangle = Rectangle(
                (index, 0),
                1,
                1,
                facecolor=face_color,
                edgecolor="black",
                linewidth=1.6,
            )

            self.ax.add_patch(rectangle)

            register_number = (
                len(register) - 1 - index
            )

            self.ax.text(
                index + 0.5,
                0.70,
                f"R{register_number}",
                ha="center",
                va="center",
                fontsize=8,
                fontweight="bold",
            )

            self.ax.text(
                index + 0.5,
                0.30,
                str(bit),
                ha="center",
                va="center",
                fontsize=16,
                fontweight="bold",
            )

            # タップ位置を赤丸で表示
            if index in self.taps:
                self.ax.plot(
                    index + 0.5,
                    1.12,
                    marker="o",
                    markersize=9,
                    color="red",
                )

                self.ax.text(
                    index + 0.5,
                    1.34,
                    "tap",
                    ha="center",
                    fontsize=8,
                    color="red",
                )

        # ---------------------------------------------
        # フィードバックと出力情報
        # ---------------------------------------------

        feedback = self.feedbacks[self.index]
        output_bit = self.outputs[self.index]
        tap_values = self.tap_values[self.index]

        information_parts = []

        if feedback is not None:
            information_parts.append(
                f"feedback={feedback}"
            )

        if output_bit is not None:
            information_parts.append(
                f"output={output_bit}"
            )

        if tap_values is not None:
            xor_expression = " xor ".join(
                map(str, tap_values)
            )

            information_parts.append(
                f"tap XOR: {xor_expression} = {feedback}"
            )

        center_x = shown_count / 2

        if information_parts:
            self.ax.text(
                center_x,
                1.68,
                "    ".join(information_parts),
                ha="center",
                fontsize=11,
            )
        else:
            self.ax.text(
                center_x,
                1.68,
                (
                    "Initial state. "
                    "Press Play, Next, or Run 100."
                ),
                ha="center",
                fontsize=11,
            )

        polynomial_text = polynomial_to_string(
            self.exponents
        )

        title = (
            f"LFSR clock {self.index}/{STEPS}    "
            f"G(x) = {polynomial_text}    "
            f"({note})"
        )

        self.ax.set_title(
            title,
            fontsize=12,
        )

        self.ax.text(
            center_x,
            -0.35,
            (
                "1 = gray, 0 = white. "
                "Feedback enters from the left; "
                "R0 exits from the right."
            ),
            ha="center",
            fontsize=10,
        )

        margin = max(
            1.0,
            shown_count * 0.05,
        )

        self.ax.set_xlim(
            -margin,
            shown_count + margin,
        )

        self.ax.set_ylim(
            -0.65,
            1.95,
        )

        self.ax.axis("off")
        self.canvas.draw_idle()

    # =====================================================
    # テキスト情報の更新
    # =====================================================

    def update_info(self):

        polynomial_text = polynomial_to_string(
            self.exponents
        )

        seed_text = bits_to_str(self.seed)

        self.info_var.set(
            f"degree={self.degree} | "
            f"G(x)={polynomial_text} | "
            f"taps={self.taps} | "
            f"seed={seed_text} | "
            f"theoretical period="
            f"2^{self.degree}-1="
            f"{self.theoretical_period} | "
            f"steps={STEPS}"
        )

        if self.index == 0:
            message = (
                "Initial state.\n"
                "Press Play, Next, or Run 100.\n"
            )
        else:
            before = bits_to_str(
                self.states[self.index - 1]
            )

            after = bits_to_str(
                self.states[self.index]
            )

            tap_values = self.tap_values[self.index]

            if tap_values is None:
                tap_text = ""
            else:
                tap_text = " xor ".join(
                    map(str, tap_values)
                )

            message = (
                f"Clock {self.index}\n"
                f"before   : {before}\n"
                f"after    : {after}\n"
                f"tap bits : {tap_text}\n"
                f"output   : "
                f"{self.outputs[self.index]}\n"
                f"feedback : "
                f"{self.feedbacks[self.index]}\n"
            )

        self.text.delete(
            "1.0",
            tk.END,
        )

        self.text.insert(
            tk.END,
            message,
        )

        if self.index == STEPS:
            self.text.insert(
                tk.END,
                (
                    "\nFinished 100 steps. "
                    "Statistics are shown below.\n\n"
                ),
            )

            self.text.insert(
                tk.END,
                self.stats_text(),
            )

    # =====================================================
    # 再生処理
    # =====================================================

    def play(self):
        """アニメーションを再生する。"""

        if self.index >= STEPS:
            self.set_index(0)

        if self.playing:
            return

        self.playing = True
        self.schedule_next()

    def pause(self):
        """アニメーションを停止する。"""

        self.playing = False

        if self.after_id is not None:
            try:
                self.root.after_cancel(
                    self.after_id
                )
            except tk.TclError:
                pass

            self.after_id = None

    def toggle_play(self):
        """再生と停止を切り替える。"""

        if self.playing:
            self.pause()
        else:
            self.play()

    def schedule_next(self):
        """次のクロック更新を予約する。"""

        if not self.playing:
            return

        if self.after_id is not None:
            try:
                self.root.after_cancel(
                    self.after_id
                )
            except tk.TclError:
                pass

            self.after_id = None

        self.after_id = self.root.after(
            self.interval_ms,
            self.play_tick,
        )

    def play_tick(self):
        """再生中の1クロック分の処理。"""

        self.after_id = None

        if not self.playing:
            return

        if self.index < STEPS:
            self.set_index(
                self.index + 1
            )
            self.schedule_next()
        else:
            self.pause()
            self.show_stats()

    # =====================================================
    # ボタン処理
    # =====================================================

    def next_step(self):
        """1クロック進める。"""

        self.pause()

        if self.index < STEPS:
            self.set_index(
                self.index + 1
            )

        if self.index == STEPS:
            self.show_stats()

    def prev_step(self):
        """1クロック戻る。"""

        self.pause()

        if self.index > 0:
            self.set_index(
                self.index - 1
            )

    def reset(self):
        """初期状態に戻る。"""

        self.pause()
        self.set_index(0)

    def run_to_end(self):
        """100クロック目まで一気に進める。"""

        self.pause()
        self.set_index(STEPS)
        self.show_stats()

    # =====================================================
    # 統計情報
    # =====================================================

    def stats_text(self):
        """統計情報を文字列として作成する。"""

        stats = compute_stats(
            states=self.states,
            feedbacks=self.feedbacks,
            output_sequence=self.output_sequence,
            seed=self.seed,
            theoretical_period=self.theoretical_period,
        )

        first_return = stats[
            "first_return_clock"
        ]

        if first_return is None:
            first_return_text = (
                "not observed in 100 steps"
            )
        else:
            first_return_text = str(
                first_return
            )

        output_sequence_text = bits_to_str(
            self.output_sequence
        )

        if len(output_sequence_text) > 120:
            output_sequence_display = (
                output_sequence_text[:120]
                + "..."
            )
        else:
            output_sequence_display = (
                output_sequence_text
            )

        return (
            "Statistics\n"
            "----------\n"
            f"degree                  : "
            f"{self.degree}\n"
            f"polynomial              : "
            f"{polynomial_to_string(self.exponents)}\n"
            f"taps                    : "
            f"{self.taps}\n"
            f"seed                    : "
            f"{bits_to_str(self.seed)}\n"
            f"steps                   : "
            f"{stats['steps']}\n"
            f"output 0 count          : "
            f"{stats['output_0']}\n"
            f"output 1 count          : "
            f"{stats['output_1']}\n"
            f"output 1 ratio          : "
            f"{stats['output_1_ratio']:.3f}\n"
            f"feedback 0 count        : "
            f"{stats['feedback_0']}\n"
            f"feedback 1 count        : "
            f"{stats['feedback_1']}\n"
            f"unique states observed  : "
            f"{stats['unique_states']}\n"
            f"first return to seed    : "
            f"{first_return_text}\n"
            f"theoretical period      : "
            f"{stats['theoretical_period']}\n"
            f"run count               : "
            f"{stats['run_count']}\n"
            f"max run length          : "
            f"{stats['max_run']}\n"
            f"average run length      : "
            f"{stats['avg_run']:.3f}\n"
            f"output sequence         : "
            f"{output_sequence_display}\n"
        )

    def show_stats(self):
        """統計情報を下部テキスト欄に表示する。"""

        self.pause()

        self.text.delete(
            "1.0",
            tk.END,
        )

        self.text.insert(
            tk.END,
            self.stats_text(),
        )

    # =====================================================
    # 新しいLFSR問題の生成
    # =====================================================

    def new_problem(self):
        """多項式、タップ、初期値を新しく生成する。"""

        self.pause()

        (
            self.degree,
            self.exponents,
        ) = choose_random_primitive_polynomial()

        self.taps = polynomial_to_taps(
            self.degree,
            self.exponents,
        )

        self.seed = random_seed_bits(
            self.degree
        )

        self.theoretical_period = (
            2 ** self.degree - 1
        )

        (
            self.states,
            self.feedbacks,
            self.outputs,
            self.tap_values,
            self.output_sequence,
        ) = make_history(
            self.seed,
            self.taps,
            STEPS,
        )

        self.set_index(0)

    # =====================================================
    # 終了処理
    # =====================================================

    def close(self):
        """アプリケーションを終了する。"""

        self.pause()

        try:
            plt.close(self.fig)
        except Exception:
            pass

        try:
            self.root.quit()
            self.root.destroy()
        except tk.TclError:
            pass


# =========================================================
# 8. メイン処理
# =========================================================

def main():
    root = tk.Tk()

    # ttkテーマの設定
    style = ttk.Style(root)

    available_themes = style.theme_names()

    if "vista" in available_themes:
        style.theme_use("vista")
    elif "clam" in available_themes:
        style.theme_use("clam")

    LFSRApp(root)

    root.mainloop()


if __name__ == "__main__":
    main()